# 01 — Export | Web Platform Data

**Project:** MLB Performance Intelligence Platform

Exports the Gold layer to one compact JSON file that powers the static web platform in `web/`
(team, player, head-to-head, leaders and Dominican players explorer). Tables are written in a
columnar layout (`{"cols": [...], "rows": [[...]]}`) to keep the file small.

| Output | Location |
|---|---|
| `mlb_web_data.json` | `/Volumes/mlb_intelligence/gold/exports/` (Databricks) |

Build the site with `python web/build.py <path-to>/mlb_web_data.json` → `web/dist/index.html`.

In [ ]:
import sys, importlib, json, os
_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + _nb.rsplit("/", 2)[0] + "/shared")
import config as C
import pipeline as P
for _m in (C, P):
    importlib.reload(_m)
from datetime import date, datetime, timezone
from decimal import Decimal

RUN_ID = P.new_run_id()
T = lambda s, n: C.table(s, n)
EXPORT_DIR = os.environ.get("MLB_EXPORT_DIR", f"/Volumes/{C.CATALOG}/gold/exports")

def _v(x):
    if isinstance(x, (date, datetime)):
        return x.isoformat()[:10]
    if isinstance(x, Decimal):
        return float(x)
    if isinstance(x, float):
        return round(x, 3)
    return x

def table_json(sql):
    df = spark.sql(sql)
    return {"cols": df.columns, "rows": [[_v(v) for v in r] for r in df.collect()]}

def grouped(sql, key_cols):
    """Rows grouped into {"key": [[...], ...]} for per-entity time series."""
    df = spark.sql(sql)
    cols = [c for c in df.columns if c not in key_cols]
    out = {}
    for r in df.collect():
        out.setdefault("|".join(str(r[k]) for k in key_cols), []).append([_v(r[c]) for c in cols])
    return {"cols": cols, "series": out}

In [ ]:
with P.log_step(spark, RUN_ID, "export", "web_platform_export") as step:
    data = {
        "meta": {
            "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "last_game_date": _v(spark.sql(f"SELECT MAX(official_date) FROM {T('silver', 'fact_game')} WHERE is_completed").collect()[0][0]),
            "seasons": [r[0] for r in spark.sql(f"SELECT DISTINCT season FROM {T('gold', 'team_season_summary')} ORDER BY 1").collect()],
        },
        "teams": table_json(f"""SELECT team_id, team_name, team_abbreviation, league_abbreviation, division_name
                                FROM {T('gold', 'dim_team')} ORDER BY team_name"""),
        "team_seasons": table_json(f"""SELECT season, season_type, team_id, team_abbreviation, league_abbreviation, division_name,
              games, wins, losses, win_pct, runs_scored, runs_allowed, run_differential, runs_scored_per_game,
              runs_allowed_per_game, pythagorean_win_pct, expected_wins, luck_wins, home_wins, home_losses, away_wins,
              away_losses, one_run_wins, one_run_losses, extra_inning_wins, extra_inning_losses, last_10_wins,
              last_10_losses, current_streak, longest_win_streak, longest_losing_streak, avg_home_attendance,
              division_rank, games_back FROM {T('gold', 'team_season_summary')}"""),
        "team_logs": grouped(f"""SELECT season, season_type, team_id, team_game_number, official_date, opponent_id, is_home,
              runs_scored, runs_allowed, result, wins_to_date, losses_to_date, run_differential_to_date, wins_last_10
              FROM {T('gold', 'team_game_log')} ORDER BY season, season_type, team_id, team_game_number""",
              ["season", "season_type", "team_id"]),
        "head_to_head": table_json(f"""SELECT season, season_type, team_id, opponent_id, games, wins, losses,
              runs_scored, runs_allowed, first_meeting, last_meeting, last_result FROM {T('gold', 'head_to_head')}"""),
        "games": table_json(f"""SELECT game_pk, season, season_type, official_date, home_team_id, away_team_id, home_score,
              away_score, venue_name, innings_played, series_description, winning_pitcher_id, losing_pitcher_id
              FROM {T('gold', 'fact_game')} WHERE is_completed ORDER BY official_date, game_pk"""),
        "players": table_json(f"""SELECT player_id, player_name, birth_country, birth_city, primary_position, primary_position_name, bats, throws,
              birth_date, height_in, weight_lb, mlb_debut_date, current_team_id, is_active
              FROM {T('gold', 'dim_player')} p
              WHERE player_id IN (SELECT player_id FROM {T('gold', 'player_season_batting')}
                                  UNION SELECT player_id FROM {T('gold', 'player_season_pitching')})"""),
        "batting": table_json(f"""SELECT season, season_type, player_id, team_id, teams_played, age, games, games_started, pa, ab,
              r, h, doubles, triples, hr, rbi, bb, ibb, so, hbp, sb, cs, sf, tb, gidp, avg, obp, slg, ops, iso, babip,
              k_pct, bb_pct, is_qualified, ops_rank_qualified, avg_rank_qualified, hr_rank, rbi_rank, sb_rank
              FROM {T('gold', 'player_season_batting')} WHERE pa > 0"""),
        "pitching": table_json(f"""SELECT season, season_type, player_id, team_id, teams_played, age, role, games, games_started,
              w, l, sv, svo, hld, bs, cg, sho, quality_starts, ip_display, outs, batters_faced, pitches, h, r, er, bb,
              so, hr, hbp, era, whip, k_per_9, bb_per_9, hr_per_9, k_bb_ratio, k_pct, bb_pct, fip, is_qualified,
              era_rank_qualified, so_rank, w_rank, sv_rank FROM {T('gold', 'player_season_pitching')}"""),
        # Game logs only for regulars (>= 100 PA / >= 20 IP) and with compact columns, to keep the file small
        "batting_logs": grouped(f"""SELECT g.season, g.season_type, g.player_id, DATE_FORMAT(g.official_date, 'MM-dd') AS game_day,
              g.opponent_id, g.ab, g.h, g.hr, g.rbi, g.ops_last_15, g.ops_season_to_date
              FROM {T('gold', 'batting_game_log')} g
              JOIN {T('gold', 'player_season_batting')} s USING (season, season_type, player_id)
              WHERE g.season_type = 'Regular Season' AND s.pa >= 100 ORDER BY g.season, g.player_id, g.player_game_number""",
              ["season", "season_type", "player_id"]),
        "pitching_logs": grouped(f"""SELECT g.season, g.season_type, g.player_id, DATE_FORMAT(g.official_date, 'MM-dd') AS game_day,
              g.opponent_id, g.innings_pitched_display, g.er, g.so, g.decision, g.era_last_5, g.era_season_to_date
              FROM {T('gold', 'pitching_game_log')} g
              JOIN {T('gold', 'player_season_pitching')} s USING (season, season_type, player_id)
              WHERE g.season_type = 'Regular Season' AND s.outs >= 60 ORDER BY g.season, g.player_id, g.appearance_number""",
              ["season", "season_type", "player_id"]),
        "countries": table_json(f"""SELECT season, season_type, birth_country, players, hitters, pitchers, h, hr, rbi, sb,
              pitcher_wins, saves, pitcher_strikeouts, era, pct_of_players, pct_of_home_runs, pct_of_strikeouts,
              country_rank_by_players FROM {T('gold', 'country_season_summary')}"""),
        "latest_lines": table_json(f"""SELECT line_type, official_date, player_id, player_name, birth_country, team_abbreviation,
              opponent, position, line FROM {T('gold', 'latest_game_day_lines')}
              ORDER BY line_type, team_abbreviation, player_name"""),
        "daily": table_json(f"""SELECT game_date, season, season_type, completed_games, postponed_games, makeup_games, total_runs,
              avg_runs_per_game, home_runs, avg_attendance, avg_duration_minutes FROM {T('gold', 'league_daily_summary')}
              ORDER BY game_date"""),
        "quality": table_json(f"""SELECT check_id, check_name, severity, status, failed_rows, checked_rows, checked_at
              FROM {C.DQ_RESULTS} WHERE run_id = (SELECT MAX_BY(run_id, checked_at) FROM {C.DQ_RESULTS}) ORDER BY check_id"""),
    }
    os.makedirs(EXPORT_DIR, exist_ok=True)
    path = os.path.join(EXPORT_DIR, "mlb_web_data.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, separators=(",", ":"))
    step.records = sum(len(v.get("rows", v.get("series", []))) for v in data.values() if isinstance(v, dict))
    step.message = f"{path} ({os.path.getsize(path) / 1e6:.1f} MB)"
    print(step.message)